# Lecture 16: NLP Applications & Case Studies
### NLP Course 2027

---

## Learning Outcomes
- Implement BM25 and semantic search
- Apply LDA for topic modeling
- Build extractive and abstractive summarizers
- Create a content-based recommender using NLP

**Primary Reference:** *Practical NLP* Ch.6, 7, 10

## 1. Information Retrieval & Search

### Classical IR: BM25

BM25 (Best Match 25) is the gold standard for keyword-based retrieval:

```
BM25(q, d) = sum over terms t:
    IDF(t) * (tf(t,d) * (k1+1)) / (tf(t,d) + k1*(1 - b + b*|d|/avgdl))

  IDF(t): how rare is term t across all docs
  tf(t,d): how often term t appears in doc d
  k1=1.5, b=0.75: tuning parameters
```

### Semantic Search
BM25 matches keywords. Semantic search matches **meaning**:
- 'car' retrieves docs about 'automobile'
- 'happy' retrieves docs about 'joyful'
- Uses dense embeddings + vector similarity

In [1]:
# BM25 from scratch (simplified)
import math
from collections import Counter

class BM25:
    """Simplified BM25 retrieval model."""
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = []
        self.df = Counter()  # document frequency
        self.idf = {}
        self.avgdl = 0

    def fit(self, documents):
        """documents: list of tokenized docs (list of words)"""
        self.docs = documents
        N = len(documents)
        self.avgdl = sum(len(d) for d in documents) / N
        # Count document frequencies
        for doc in documents:
            for term in set(doc):
                self.df[term] += 1
        # Compute IDF
        for term, df in self.df.items():
            self.idf[term] = math.log((N - df + 0.5) / (df + 0.5) + 1)
        return self

    def score(self, query_terms, doc_idx):
        """Compute BM25 score for one doc."""
        doc = self.docs[doc_idx]
        dl = len(doc)
        tf_counter = Counter(doc)
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = tf_counter[term]
            idf = self.idf[term]
            score += idf * (tf * (self.k1 + 1)) / (
                tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            )
        return score

    def search(self, query, top_k=5):
        query_terms = query.lower().split()
        scores = [(i, self.score(query_terms, i)) for i in range(len(self.docs))]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# Demo
corpus = [
    'machine learning algorithms for text classification',
    'deep learning neural networks and NLP',
    'natural language processing with transformers and BERT',
    'information retrieval and search engines',
    'topic modeling with LDA and gensim',
    'text classification sentiment analysis reviews',
    'transformer architecture self attention mechanism',
]
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25().fit(tokenized)

query = 'deep learning transformer NLP'
results = bm25.search(query, top_k=3)
print(f'Query: "{query}"\nTop results:')
for rank, (idx, score) in enumerate(results, 1):
    print(f'  [{rank}] {score:.4f}  {corpus[idx]}')

Query: "deep learning transformer NLP"
Top results:
  [1] 4.4118  deep learning neural networks and NLP
  [2] 1.7737  transformer architecture self attention mechanism
  [3] 1.1376  machine learning algorithms for text classification


In [2]:
# Semantic search with sentence-transformers
# !pip install sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    model = SentenceTransformer('all-MiniLM-L6-v2')  # 80MB, fast

    # Encode corpus
    corpus_embeddings = model.encode(corpus, show_progress_bar=False)

    def semantic_search(query, corpus_embs, corpus_docs, top_k=3):
        query_emb = model.encode([query])
        sims = cosine_similarity(query_emb, corpus_embs)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(corpus_docs[i], sims[i]) for i in top_idx]

    query = 'self-attention mechanism in neural networks'
    print(f'Semantic search: "{query}"')
    for doc, score in semantic_search(query, corpus_embeddings, corpus):
        print(f'  [{score:.4f}] {doc}')

except ImportError:
    print('Install: pip install sentence-transformers')
    print('Semantic search finds conceptually similar docs, not just keyword matches')

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semantic search: "self-attention mechanism in neural networks"
  [0.6310] transformer architecture self attention mechanism
  [0.3703] deep learning neural networks and NLP
  [0.2283] machine learning algorithms for text classification


## 2. Topic Modeling

**Topic modeling** discovers latent themes in a document collection.

### LDA (Latent Dirichlet Allocation)
```
Assumptions:
  - A corpus is a mixture of K topics
  - Each document is a mixture of topics (document-topic distribution)
  - Each topic is a distribution over words (topic-word distribution)

Generative story:
  For each document d:
    Draw topic mixture: theta_d ~ Dirichlet(alpha)
    For each word in d:
      Draw topic: z ~ Multinomial(theta_d)
      Draw word:  w ~ Multinomial(phi_z)
```

In [3]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from nltk.corpus import reuters
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

nltk.download('reuters', quiet=True)
nltk.download('stopwords', quiet=True)

# Prepare corpus
stop_words = set(stopwords.words('english'))

def preprocess(text):
    return [w.lower() for w in word_tokenize(text)
            if w.isalpha() and w.lower() not in stop_words and len(w) > 3]

# Use a subset of Reuters
print('Loading Reuters corpus...')
docs = [preprocess(reuters.raw(fid)) for fid in reuters.fileids()[:500]]
print(f'Docs: {len(docs)}, Sample: {docs[0][:10]}')

# Build vocabulary
dictionary = corpora.Dictionary(docs)
dictionary.filter_extremes(no_below=5, no_above=0.5)
bow_corpus = [dictionary.doc2bow(doc) for doc in docs]

# Train LDA
print('Training LDA...')
lda = LdaModel(
    bow_corpus, num_topics=8, id2word=dictionary,
    passes=5, alpha='auto', eta='auto', random_state=42
)
print('Done!')

Loading Reuters corpus...
Docs: 500, Sample: ['asian', 'exporters', 'fear', 'damage', 'rift', 'mounting', 'trade', 'friction', 'japan', 'raised']
Training LDA...
Done!


In [4]:
# Show topics
print('LDA Topics:')
print('=' * 60)
for topic_id in range(8):
    words = lda.show_topic(topic_id, topn=8)
    word_str = ', '.join(f'{w}({p:.3f})' for w, p in words)
    print(f'Topic {topic_id}: {word_str}')
    print()

LDA Topics:
Topic 0: prices(0.014), price(0.011), company(0.010), unit(0.010), year(0.009), tonnes(0.009), corp(0.008), industrial(0.008)

Topic 1: today(0.016), market(0.014), billion(0.013), bank(0.013), futures(0.011), ministry(0.009), money(0.009), unchanged(0.009)

Topic 2: tonnes(0.038), april(0.016), trade(0.015), sugar(0.014), report(0.014), wheat(0.013), japan(0.012), record(0.012)

Topic 3: dlrs(0.018), billion(0.017), year(0.015), days(0.014), shares(0.014), trade(0.013), would(0.013), government(0.011)

Topic 4: dollar(0.015), bank(0.013), japan(0.012), rates(0.010), market(0.010), meeting(0.010), exchange(0.010), economic(0.009)

Topic 5: dlrs(0.023), year(0.018), would(0.015), deficit(0.013), billion(0.011), last(0.009), market(0.008), growth(0.008)

Topic 6: billion(0.035), dlrs(0.027), year(0.025), last(0.014), first(0.010), quarter(0.010), company(0.009), market(0.009)

Topic 7: dlrs(0.062), loss(0.037), company(0.033), share(0.019), year(0.019), profit(0.019), stock(0

In [5]:
# Infer topic distribution for a new document
new_doc = 'oil prices rose sharply after OPEC announced production cuts'
new_bow = dictionary.doc2bow(preprocess(new_doc))
topic_dist = lda.get_document_topics(new_bow)

print(f'Document: "{new_doc}"')
print('Topic distribution:')
for topic_id, prob in sorted(topic_dist, key=lambda x: x[1], reverse=True):
    topic_words = [w for w, p in lda.show_topic(topic_id, topn=5)]
    print(f'  Topic {topic_id} ({prob:.4f}): {topic_words}')

Document: "oil prices rose sharply after OPEC announced production cuts"
Topic distribution:
  Topic 2 (0.5267): ['tonnes', 'april', 'trade', 'sugar', 'report']
  Topic 0 (0.4033): ['prices', 'price', 'company', 'unit', 'year']
  Topic 7 (0.0172): ['dlrs', 'loss', 'company', 'share', 'year']
  Topic 4 (0.0120): ['dollar', 'bank', 'japan', 'rates', 'market']
  Topic 6 (0.0119): ['billion', 'dlrs', 'year', 'last', 'first']
  Topic 5 (0.0105): ['dlrs', 'year', 'would', 'deficit', 'billion']
  Topic 3 (0.0102): ['dlrs', 'billion', 'year', 'days', 'shares']


## 3. Text Summarization

### Two Approaches

| Approach | Method | Pros | Cons |
|----------|--------|------|------|
| **Extractive** | Select key sentences | Faithful, fast | May miss big picture |
| **Abstractive** | Generate new text | Concise, flexible | Can hallucinate |

### TextRank (Extractive)
- Represent each sentence as a TF-IDF vector
- Build graph: nodes=sentences, edges=cosine similarity
- Run PageRank; highest-scoring sentences = summary

In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

def textrank_summarize(text, num_sentences=3):
    """Extractive summarization using TextRank algorithm."""
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    # Build TF-IDF sentence vectors
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(sentences)

    # Build similarity graph
    sim_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(sim_matrix, 0)

    # PageRank-style scoring
    d = 0.85  # damping factor
    scores = np.ones(len(sentences))
    for _ in range(50):  # iterate
        new_scores = (1 - d) + d * sim_matrix.T @ scores
        if np.allclose(scores, new_scores, atol=1e-6):
            break
        scores = new_scores

    # Select top-N sentences in original order
    top_indices = sorted(np.argsort(scores)[-num_sentences:].tolist())
    return ' '.join(sentences[i] for i in top_indices)

text = """
Natural Language Processing (NLP) is a branch of artificial intelligence
that focuses on enabling computers to understand, interpret, and generate human language.
The field has undergone a dramatic transformation over the past decade.
Early NLP systems relied heavily on hand-crafted rules and linguistic knowledge.
These rule-based systems were brittle and hard to scale.
The introduction of statistical methods in the 1990s marked a major shift.
Machine learning algorithms replaced hand-crafted rules in many applications.
The real revolution came with deep learning and especially transformer models.
BERT and GPT demonstrated that pretraining on large text corpora could yield
remarkable improvements across all NLP tasks.
Today, large language models like ChatGPT and Claude can engage in fluent conversations,
write code, answer complex questions, and even generate creative content.
"""
summary = textrank_summarize(text, num_sentences=3)
print('Original length:', len(text.split()), 'words')
print('Summary length: ', len(summary.split()), 'words')
print('\nSummary:')
print(summary)

Original length: 126 words
Summary length:  43 words

Summary:
Early NLP systems relied heavily on hand-crafted rules and linguistic knowledge. Machine learning algorithms replaced hand-crafted rules in many applications. Today, large language models like ChatGPT and Claude can engage in fluent conversations,
write code, answer complex questions, and even generate creative content.


In [7]:
# Abstractive summarization with BART
from transformers import pipeline

summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
long_text = """
The development of large language models has transformed the field of natural language
processing. Starting with BERT in 2018, which introduced bidirectional pretraining,
and continuing through GPT-3, ChatGPT, and beyond, these models have achieved
human-level performance on many language understanding benchmarks. The key innovation
was training on massive text corpora using self-supervised objectives like masked
language modeling and next token prediction. This allows models to learn rich
representations of language without requiring expensive labeled data. However, these
models also raise important questions about bias, factuality, environmental costs,
and potential misuse. Researchers are actively working on making these systems more
faithful, efficient, and equitable.
"""
summary = summarizer(long_text, max_length=80, min_length=30, do_sample=False)
print('Abstractive summary:')
print(summary[0]['summary_text'])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Abstractive summary:
The development of large language models has transformed the field of natural language processing. These models have achieved human-level performance on many language understanding benchmarks. But these models also raise important questions about bias, factuality, environmental costs, and potential misuse.


## 4. NLP-Powered Recommendation

Content-based filtering uses item descriptions to recommend similar items.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Product catalog
catalog = [
    {'id': 1, 'name': 'Intro to Machine Learning',
     'desc': 'Beginner course covering supervised learning, classification, regression, and evaluation.'},
    {'id': 2, 'name': 'Deep Learning Fundamentals',
     'desc': 'Neural networks, backpropagation, CNNs, RNNs, LSTMs, and modern architectures.'},
    {'id': 3, 'name': 'Natural Language Processing',
     'desc': 'Text classification, NER, transformer models, BERT, fine-tuning and deployment.'},
    {'id': 4, 'name': 'Computer Vision',
     'desc': 'Image classification, object detection, CNNs, transfer learning with ResNet and VGG.'},
    {'id': 5, 'name': 'Data Science with Python',
     'desc': 'Pandas, NumPy, matplotlib, scikit-learn, exploratory data analysis.'},
    {'id': 6, 'name': 'NLP with Transformers',
     'desc': 'Hugging Face transformers, BERT, GPT, fine-tuning, text generation and summarization.'},
    {'id': 7, 'name': 'Reinforcement Learning',
     'desc': 'Q-learning, policy gradients, OpenAI Gym, reward modeling, multi-agent systems.'},
]

descriptions = [item['desc'] for item in catalog]
names = [item['name'] for item in catalog]

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(descriptions)
sim_matrix = cosine_similarity(tfidf_matrix)

def recommend(item_id, top_k=3):
    idx = item_id - 1
    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][1:top_k+1]  # skip self
    return [(names[i], sims[i]) for i in top_idx]

for item in catalog[:3]:
    print(f'If you liked: {item["name"]}')
    print('You might also like:')
    for name, score in recommend(item['id']):
        print(f'  - {name}  (similarity: {score:.3f})')
    print()

If you liked: Intro to Machine Learning
You might also like:
  - Computer Vision  (similarity: 0.137)
  - Natural Language Processing  (similarity: 0.071)
  - Reinforcement Learning  (similarity: 0.062)

If you liked: Deep Learning Fundamentals
You might also like:
  - Computer Vision  (similarity: 0.090)
  - Reinforcement Learning  (similarity: 0.000)
  - NLP with Transformers  (similarity: 0.000)

If you liked: Natural Language Processing
You might also like:
  - NLP with Transformers  (similarity: 0.346)
  - Intro to Machine Learning  (similarity: 0.071)
  - Computer Vision  (similarity: 0.067)



## Practice Exercises

See **`Lecture-16-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Application | Technique | Key Tool |
|-------------|-----------|----------|
| Keyword search | BM25 | rank-bm25 / gensim |
| Semantic search | Dense embeddings | sentence-transformers |
| Topic modeling | LDA | gensim |
| Extractive summarization | TextRank | custom / sumy |
| Abstractive summarization | Seq2seq | BART, T5 (HF) |
| Content recommendation | TF-IDF similarity | scikit-learn |

**Next Lecture**: NLP in Industry Domains.

---
*Book reference: Practical NLP Ch.6, 7, 10*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**